# télos MDLM: Master Overnight Training Suite
This notebook executes the target overnight training pipeline in sequence using native MLX Metal acceleration.

### Target Ratios & Pipeline:
1. **50M 1:20** (Upscaled from existing `phase_b_25m_1to20_mlx` checkpoint)
2. **25M 1:25** (Trained from scratch to 852M tokens)
3. **50M 1:25** (Upscaled from newly trained `phase_b_25m_1to25_mlx` checkpoint)

Memory GC is enforced on Metal GPU to keep RAM usage strictly under 8GB.


In [1]:
import os
import sys
import time
import gc
import yaml
import math
import io
from pathlib import Path
import numpy as np

# Ensure working directory is project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

import mlx.core as mx
import mlx.nn as nn
from telos.model.mlx_components import MLXTelosTransformer, load_upscaled_weights
from telos.training.trainer import TelosMLXTrainer
from telos.data.tokenizer import load_tokenizer

def run_training_step(config_path, run_name, upscaled_source=None):
    print("=" * 85)
    print("STARTING RUN: " + str(config_path))
    print("=" * 85)
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    model = MLXTelosTransformer(**cfg["model"])
    model.set_dtype(mx.bfloat16)
    if upscaled_source:
        src_ckpt, src_cfg = upscaled_source
        print("  [Net2Net] Upscaling model weights from: " + str(src_ckpt))
        load_upscaled_weights(model, cfg["model"], src_ckpt, src_cfg)
    trainer = TelosMLXTrainer(model, cfg)
    trainer.train()
    del model, trainer
    gc.collect()
    mx.clear_cache()
    print("FINISHED RUN: " + str(config_path) + "\n")

# PIPELINE DEFINITION
start_time = time.time()

# 3. 50M 1:25 (Upscaled from new 25M 1:25 model.safetensors)
run_training_step(
    "configs/phase_b_50m_1to25_mlx.yaml",
    "50M_1to25_Ratio_Upscaled",
    upscaled_source=("checkpoints/phase_b_25m_1to25_mlx/model.safetensors", "configs/phase_b_25m_1to25_mlx.yaml")
)

total_elapsed = (time.time() - start_time) / 3600.0
print("=" * 85)
print(f"RUN 3 COMPLETED SUCCESSFULLY IN {total_elapsed:.2f} HOURS!")
print("=" * 85)


STARTING RUN: configs/phase_b_50m_1to25_mlx.yaml
  [Net2Net] Upscaling model weights from: checkpoints/phase_b_25m_1to25_mlx/model.safetensors
  [Upscaling] Loading source config: configs/phase_b_25m_1to25_mlx.yaml
  [Upscaling] Loading source weights: checkpoints/phase_b_25m_1to25_mlx/model.safetensors
  [Upscaling] Depth mapping (Target <- Source): [0, 1, 2, 3, 4, 5, 6, 7]
  [Upscaling] Success: Initialized model with upscaled weights.
  Loading pre-tokenized dataset from data/python_corpus_mac.bin...


  Checkpoint Directory: checkpoints/phase_b_50m_1to25_mlx (Versioned)


  Step      1/3051 | ELBO Loss: 150.20 | CE: 9.613 | LR: 2.00e-06 |   0.3 st/s |    34,533 tok/s | Metal Unified GPU: 0.83GB (Peak: 4.51GB) | Swap: 3743.75M | ETA: 192.9m


  Step     50/3051 | ELBO Loss: 111.53 | CE: 6.115 | LR: 5.10e-05 |   0.3 st/s |    35,495 tok/s | Metal Unified GPU: 0.83GB (Peak: 4.51GB) | Swap: 3743.75M | ETA: 184.7m


  Step    100/3051 | ELBO Loss: 113.50 | CE: 6.287 | LR: 1.01e-04 |   0.3 st/s |    35,473 tok/s | Metal Unified GPU: 0.83GB (Peak: 4.51GB) | Swap: 3743.75M | ETA: 181.7m


...

  Step   3000/3051 | ELBO Loss: 124.42 | CE: 6.064 | LR: 3.02e-05 |   0.3 st/s |    34,685 tok/s | Metal Unified GPU: 0.83GB (Peak: 4.54GB) | Swap: 2641.69M | ETA:  3.2m


  [Checkpoint] Saved weights to checkpoints/phase_b_50m_1to25_mlx/checkpoint_step_3000.safetensors


  Step   3050/3051 | ELBO Loss:  84.06 | CE: 6.014 | LR: 3.00e-05 |   0.3 st/s |    34,671 tok/s | Metal Unified GPU: 0.83GB (Peak: 4.54GB) | Swap: 2641.69M | ETA:  0.1m


  Step   3051/3051 | ELBO Loss: 111.82 | CE: 6.067 | LR: 3.00e-05 |   0.3 st/s |    34,671 tok/s | Metal Unified GPU: 0.83GB (Peak: 4.54GB) | Swap: 2641.69M | ETA:  0.0m


  Training Complete! Total time: 192.24 minutes.
  Saved standalone model artifact to checkpoints/phase_b_50m_1to25_mlx/
FINISHED RUN: configs/phase_b_50m_1to25_mlx.yaml

RUN 3 COMPLETED SUCCESSFULLY IN 3.20 HOURS!
